# ML 모델 파이프라인

실무에서 사용하는 ML 파이프라인 구축법을 배웁니다.

## 학습 목표
- 전처리 파이프라인 구성
- 하이퍼파라미터 튜닝
- 모델 저장/로딩
- 실무 워크플로우

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

print("파이프라인 학습 시작!")

## 1. 데이터 준비

In [ ]:
# 타이타닉 데이터 생성 (실제 데이터처럼)
np.random.seed(42)
n_samples = 500

df = pd.DataFrame({
    'Age': np.random.randint(1, 80, n_samples),
    'Fare': np.random.uniform(10, 500, n_samples),
    'Sex': np.random.choice(['male', 'female'], n_samples),
    'Pclass': np.random.choice([1, 2, 3], n_samples),
    'Embarked': np.random.choice(['S', 'C', 'Q'], n_samples),
    'Survived': np.random.choice([0, 1], n_samples, p=[0.6, 0.4])
})

print(df.head())
print(f"\n데이터 정보:")
print(df.info())

In [ ]:
# 특성과 타겟 분리
X = df.drop('Survived', axis=1)
y = df['Survived']

# 학습/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"훈련 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")

## 2. 전처리 파이프라인

In [ ]:
# 수치형/범주형 특성 구분
numeric_features = ['Age', 'Fare']
categorical_features = ['Sex', 'Embarked', 'Pclass']

# 전처리기 정의
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 전처리 파이프라인 합치기
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("전처리기 준비 완료!")

## 3. 전체 파이프라인 구성

In [ ]:
# 전체 파이프라인: 전처리 + 모델
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# 학습
full_pipeline.fit(X_train, y_train)

# 예측
y_pred = full_pipeline.predict(X_test)

print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")
print("\n분류 리포트:")
print(classification_report(y_test, y_pred))

## 4. 하이퍼파라미터 튜닝

In [ ]:
# 하이퍼파라미터 그리드
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5, 10]
}

# Grid Search
grid_search = GridSearchCV(
    full_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최적 점수: {grid_search.best_score_:.4f}")

In [ ]:
# 최적 모델로 예측
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print(f"튜닝 후 정확도: {accuracy_score(y_test, y_pred_best):.4f}")

## 5. 여러 모델 비교

In [ ]:
# 모델 정의
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

results = {}

for name, model in models.items():
    # 파이프라인 생성
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # 학습 및 평가
    pipeline.fit(X_train, y_train)
    score = pipeline.score(X_test, y_test)
    results[name] = score
    print(f"{name}: {score:.4f}")

## 6. 모델 저장/로딩

In [ ]:
# 모델 저장
joblib.dump(best_model, 'titanic_model.pkl')
print("모델 저장 완료!")

In [ ]:
# 모델 로딩
loaded_model = joblib.load('titanic_model.pkl')

# 로딩된 모델로 예측
y_pred_loaded = loaded_model.predict(X_test)
print(f"로딩된 모델 정확도: {accuracy_score(y_test, y_pred_loaded):.4f}")

## 7. 실무 팁: 새로운 데이터 예측

In [ ]:
# 새로운 승객 데이터
new_passenger = pd.DataFrame({
    'Age': [25],
    'Fare': [100.0],
    'Sex': ['male'],
    'Pclass': [1],
    'Embarked': ['S']
})

# 예측
prediction = loaded_model.predict(new_passenger)
probability = loaded_model.predict_proba(new_passenger)

print(f"생존 예측: {'생존' if prediction[0] == 1 else '사망'}")
print(f"생존 확률: {probability[0][1]:.2%}")

## ML 파이프라인 체크리스트

✅ 데이터 탐색 (EDA)
✅ 전처리 파이프라인 구축
✅ 학습/검증/테스트 분할
✅ 모델 학습 및 비교
✅ 하이퍼파라미터 튜닝
✅ 최종 모델 평가
✅ 모델 저장 및 배포 준비